In [2]:
#pip install torch transformers accelerate

# Quantizing TinyLLAMA 
###  Comparing performances of FP16, INT8, and INT4

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype = torch.float16,
).cuda()

prompt = "How does quantization help a LLM? It "

inputs = tokenizer(prompt, return_tensors="pt").to("cuda") #Give it to the GPU for faster Neural Netowrk computation
#Note: all calculations must be on the same memory(can't do x * y, x in CPU, y in GPU)

with torch.no_grad(): #No gradient calcuations, simple inference, no backprop/grad descent/para updates
    outputs = model.generate(
        **inputs,
        max_new_tokens=100
    )
    
    
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

KeyboardInterrupt: 

In [ ]:
print(torch.cuda.memory_allocated() / 1024 **3, "GB")
print(torch.cuda.max_memory_allocated() / 1024**3, "GB")

2.5732455253601074 GB
4.6241984367370605 GB


In [ ]:
#pip install -U bitsandbytes

In [ ]:
#pip install bitsandbytes
import torch
from transformers import BitsAndBytesConfig, AutoModelForCausalLM, AutoTokenizer

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True
)

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto" #auto decides to put it in the CPU or GPU, or somewhere else
)

prompt = "How does quantization help a LLM? It "

inputs = tokenizer(prompt, return_tensors="pt")
#Note: all calculations must be on the same memory(can't do x * y, x in CPU, y in GPU)

inputs = {k : v.to(model.device) for k, v in inputs.items()}

with torch.inference_mode(): #No gradient calcuations, simple inference, no backprop/grad descent/para updates
    outputs = model.generate(
        **inputs,
        max_new_tokens=100
    )
    
    
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

How does quantization help a LLM? It 
quantizes the input to a fixed-point format, which is more efficient for LLMs. This 
ensures that the LLM can process the input in a fixed-point format, which is more 
efficient for LLMs. This is because fixed-point formats are more efficient for 
computing operations such as addition, subtraction, and multiplication. 

In summary, quantization helps a LLM by converting the input to a fixed-point


In [ ]:
print(torch.cuda.memory_allocated() / 1024 **3, "GB")
print(torch.cuda.max_memory_allocated() / 1024**3, "GB")

1.2864537239074707 GB
4.6241984367370605 GB


# Implementing GPTQ
## Demonstrating how weights change layer-by-layer
### and how other layers compensate for the loss with the altered weights.

Steps:
- Take trained weights W
- Estimate Hessian H
- For each column:
    Quantize column
    Compute quantization error
    Use H⁻¹ to compensate remaining columns
- Repeat

In [1]:
import torch

W = torch.randn(128, 128) #Weight matrix
X = torch.randn(128, 128) #Activation matrix

W_before = W
H = X @ X.T #hopefully this works w/o numpy??
H += 1e-4 * torch.eye(H.shape[0]) #stability, dunno what it's for
H_inv = torch.inverse(H) ##haha, curvature


#Quantize sim
for i in range(0, W.shape[0]):
    col = W[:, i]

    scale = col.abs().max()/7
    q_col = torch.round(col/scale)
    
    col_hat = q_col * scale
    
    err = col - col_hat
    
    #Col i got damaged, change others to minimize dmg
    W[:, i+1:] -= err.unsqueeze(1) @ H_inv[i:i+1, i+1:] #actually insane equation
    
W_after = W

In [2]:
print(torch.norm(W_before))
print(torch.norm(W_after))
print(W_before)
print(W_after)

tensor(597.1011)
tensor(597.1011)
tensor([[  2.6474,  -0.8953,   0.9329,  ...,   0.6801,   0.2653,  -3.3538],
        [ -0.2868,   1.2176,   1.3755,  ...,  -4.4316,   4.9679,  -9.9304],
        [  0.2135,   0.8215,   1.3634,  ...,  -5.4702,   3.6261, -12.0336],
        ...,
        [  0.4376,   0.1406,   1.1492,  ...,  14.0492,  -9.1066,  22.8376],
        [ -1.3609,  -1.1781,  -0.3712,  ..., -16.3606,  11.9157, -30.7405],
        [ -0.2608,  -0.7729,   1.1611,  ...,   8.0096,  -5.9099,  16.9938]])
tensor([[  2.6474,  -0.8953,   0.9329,  ...,   0.6801,   0.2653,  -3.3538],
        [ -0.2868,   1.2176,   1.3755,  ...,  -4.4316,   4.9679,  -9.9304],
        [  0.2135,   0.8215,   1.3634,  ...,  -5.4702,   3.6261, -12.0336],
        ...,
        [  0.4376,   0.1406,   1.1492,  ...,  14.0492,  -9.1066,  22.8376],
        [ -1.3609,  -1.1781,  -0.3712,  ..., -16.3606,  11.9157, -30.7405],
        [ -0.2608,  -0.7729,   1.1611,  ...,   8.0096,  -5.9099,  16.9938]])
